In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np      
import pandas as pd
from reservoirpy.datasets import narma
from tasks import make_static_sin_task
from encoding_masking import build_masked_input, compute_tdm_params
from Reservoirs.ELM_static import run_elm_static, ELMStaticParams
from readout import split_states_targets, fit_ridge_readout, predict_readout
from metrics import mse, nrmse
from Reservoirs.LangKobayashi import expand_virtual_nodes, simulate_lk, simulate_lk_mini, extract_R_from_E, intensities_after_theta_times, charge_at_theta_intervals

In [2]:
#!pip install reservoirpy
#!pip install --upgrade scipy

In [3]:
# #sweeped params
# mask_types = ["binary", "continuous", "m_sequence", "two_sine"]
# eta_sweep = [1e-3, 3e-3, 1e-2, 3e-2, 1e-1, 3e-1]
# mask_seeds = [0, 1, 2, 3, 4]
# ridge_sweep = [0,1e-10,1e-8,1e-6,1e-4,1e-2]

In [4]:
#debugging sweep
# mask_types = ["two_sine"]
# eta_sweep = [0.01]
# mask_seeds = [42]
# ridge_sweep = [1e-6]

In [5]:
mask_types = ["m_sequence", "two_sine"]
eta_sweep = [1e-3, 3e-3, 1e-2, 3e-2, 1e-1, 3e-1]
mask_seeds = [0, 1, 2, 3, 4]
ridge_sweep = [0,1e-10,1e-8,1e-6,1e-4,1e-2]

In [6]:
#constant paramters
N = 50 # ONLY VARY WHEN WE WANNA TEST N=20 
TLk = 150
kappa = 0.1  
alpha = 0 
phi = 0 
xi = 0 
D_noise = 10**-7*0
p= 0.05
dt = 0.1 #ONLY VARY AT THE VERY END
theta = 22
tau_factor = 1.41 

washout = 30
L = 500 + washout #ONLY VARY AT THE VERY END
task_seed = 0
order = 10 #narma order
split = 0.8 # test/predict split

In [7]:
TDM_parameters = compute_tdm_params(N, theta, dt, tau_factor)
Nd_loop = TDM_parameters.Nd_loop
tap_stride = TDM_parameters.tap_stride  
Nd_delay = TDM_parameters.Nd_delay    
tau = TDM_parameters.tau

print(TDM_parameters)

TDMParams(N=50, theta=22, dt=0.1, T=1100, tau=1551.0, Nd_loop=11000, Nd_delay=15510, tap_stride=220)


In [8]:
u_full, y_full = narma(n_timesteps=L, order= order,a1=0.3, a2=0.05, b=1.5, c=0.1, seed=task_seed)

In [9]:
u_full = u_full[order:].ravel()   # align with y_full
y_full = y_full.ravel()

In [10]:
# x_norm, mask, V = build_masked_input(u_full, N=N, rng=mask_seeds,mask_type="continuous")
# Vs = expand_virtual_nodes(V, tap_stride=tap_stride, Nd=Nd_loop)  

In [11]:
# E_hist, n_hist = simulate_lk(Vs, dt, Nd_delay, alpha=alpha, kappa=kappa, phi=phi, p=p, eta=eta, D_noise=D_noise, xi=xi, Tlk=TLk, E0=1e-3+0j, n0=0.0)
# R = extract_R_from_E(E_hist, L=L, N=N, tap_stride=tap_stride, Nd_loop=Nd_loop, washout_cycles=washout)

In [12]:
results=[]

for mask_type in mask_types:
    for eta in eta_sweep:
        for mask_seed in mask_seeds:

            x_norm, mask, V = build_masked_input(u_full, N=N, rng=mask_seed, mask_type=mask_type)
            Vs = expand_virtual_nodes(V, tap_stride, Nd_loop)

            # run reservoir > get states R
            E_hist, n_hist = simulate_lk(Vs, dt, Nd_delay, alpha=alpha, kappa=kappa, phi=phi, p=p, eta=eta, D_noise=D_noise, xi=xi, Tlk=TLk, E0=1e-3+0j, n0=0.0)
            R = extract_R_from_E(E_hist, L=L, N=N, tap_stride=tap_stride, Nd_loop=Nd_loop, washout_cycles=washout)
            
            # train readout > predict y_pred
            y_target = y_full[washout:]
            s = split_states_targets(R, y_target, split=split)

            for ridge_alpha in ridge_sweep:
                model = fit_ridge_readout(s.R_train, s.y_train, alpha= ridge_alpha, fit_intercept=True) 
                y_pred = predict_readout(model, s.R_test)
            
                # compute error
                nrmse_value = nrmse(s.y_test, y_pred)

                results.append({"mask_type": mask_type, "eta": eta, "mask_seed": mask_seed,"ridge_alpha":ridge_alpha, "nrmse": nrmse_value})

df = pd.DataFrame(results)
df.to_csv("sweep_N50_results_update1.csv", index=False)

KeyboardInterrupt: 

In [13]:
df = pd.DataFrame(results)
df.to_csv("sweep_N50_results_update1.csv", index=False)

In [ ]:
print("NRMSE:", nrmse(s.y_test, y_pred))
plt.figure(figsize=(8,4))
plt.plot(s.y_test, label="True")
plt.plot(y_pred, "--", label="Pred")
#plt.title("eta=0.01 K=0.005 Tlk=10 a_ridge=0*e^-4. TEST DATA")
plt.legend()
plt.show()

In [ ]:
#df

In [ ]:
#R.shape
#y_target.shape 

In [ ]:
#df = pd.DataFrame(results)

In [ ]:
#df.to_csv("sweep_N20_results.csv", index=False)